# 10 — Transfer CNN eğitimi (Adım 10–11), Kaggle

**Ayarlar:** Accelerator **GPU T4 x2** · Internet **On** · Secrets: `GH_TOKEN` (kurulum notebook'uyla aynı).

İlk kod hücresinde adınızı/e-postanızı yazın, sonra **Save Version → Save & Run All (Commit)** seçin: notebook arka planda
çalışır, tarayıcıyı kapatabilirsiniz. Yaklaşık **1 saat** sürer (T4'te tahmini: sınıflandırma ResNet-50 ~25 dk,
EfficientNet-B0 ~15 dk, kalori modelleri ~5'er dk). Erken durdurma devreye girerse daha kısa sürer.

Bitince sonuçlar (`results.csv`, tahminler, eğitim eğrileri) **GitHub'a kendiliğinden gönderilir**; model dosyaları
repo'nun `checkpoints` Release'ine yüklenir. Bir eğitim hata verirse sonraki hücreler yine çalışır (`|| true`)
ve o ana kadarki sonuçlar gönderilir.

In [ ]:
# 1) KURULUM — her oturumda ilk bu hücre çalışır
import os, subprocess
from kaggle_secrets import UserSecretsClient

REPO = "https://github.com/traldo44/kalori-tahmini.git"
GIT_NAME, GIT_EMAIL = "Ad Soyad", "eposta@ornek.com"   # ← kendi bilgilerinizi yazın
OWNER = "ad_soyad"                                      # ← results.csv'deki owner sütunu

os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GH_TOKEN")
# Token'ı hiçbir dosyaya yazmadan git'e verir; yalnızca bu oturumun hafızasında durur.
subprocess.run(["git", "config", "--global", "credential.helper",
                '!f() { echo username=x-access-token; echo "password=$GH_TOKEN"; }; f'], check=True)

os.chdir("/kaggle/working")
if os.path.isdir("kalori-tahmini"):
    subprocess.run(["git", "-C", "kalori-tahmini", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/kaggle/working/kalori-tahmini")
subprocess.run(["git", "config", "user.name", GIT_NAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

os.environ["KALORI_DATA"] = "/tmp/kalori-data"   # veri çıktı klasörüne (/kaggle/working) girmesin
os.environ["KALORI_OWNER"] = OWNER
print("Repo hazır →", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2) VERİ — Nutrition5k (~1,4 GB) + Food-101 25 sınıflık alt küme. Aynı oturumda tekrar çalışırsa atlar.
!python scripts/download_nutrition5k.py --out /tmp/kalori-data/nutrition5k
!python scripts/get_food101.py fetch --out /tmp/kalori-data

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok → Settings → Accelerator → GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0))

## Adım 10 — sınıflandırma (Food-101, 25 sınıf) → Tablo 1

In [ ]:
!python -m src.train_cls --backbone resnet50 || true

In [ ]:
!python -m src.train_cls --backbone efficientnet_b0 || true

## Adım 11 — kalori (Nutrition5k, yalnızca RGB) → Tablo 2 ve 3
Çıta: donmuş ResNet-50 + XGBoost'un 77,4 kcal'i (Adım 9).

In [ ]:
!python -m src.train_reg --backbone resnet50 || true

In [ ]:
!python -m src.train_reg --backbone efficientnet_b0 || true

## Sonuçları gönder

In [ ]:
!python -m src.compare results/preds/cnn_xgb.csv results/preds/resnet50_reg.csv || true
!git add results && git commit -q -m "Adım 10-11: Kaggle eğitim sonuçları (ResNet-50, EfficientNet-B0)" && git pull --rebase -q && git push -q && echo "GitHub'a gönderildi ✓"
!for f in checkpoints/*.pt; do python scripts/share_checkpoint.py upload "$f"; done